# 07 - Subducted Carbonate volume 

The subducted carbonate volume is derived from the total carbonate sediment thickness that is subducting. This notebook computes the subducting carbonate sediment thickness at each trench segment and multiplies the thickness by the length of the trench segment as well as by the length of plate that has been subducted in the previous my, orthogonal to the trench.

In [ ]:
from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
import joblib
import numpy as np
import pygplates
import os, glob
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
import gplately
import gplately.grids as grids
import gplately.tools as tools
import ptt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader as shpreader
import netCDF4
from scipy import ndimage
import pandas as pd
import glob, os
from slabdip import SlabDipper
import numpy.ma as ma
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from plate_model_manager import PlateModelManager
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from cmcrameri import cm
import moviepy as mpy
plt.rcParams['font.family'] = 'Helvetica'
%matplotlib inline
# plt.style.use('ggplot')

#from pygplates_helper import *

# common variables
extent_globe = [-180, 180, -90, 90]
earth_radius = 6371.0e3
earth_surface_area = 4.0*np.pi*earth_radius**2
tessellation_threshold_radians = np.radians(0.01)

# output grid resolution - should be identical to input grid resolution!
spacingX, spacingY = 0.2, 0.2
resX, resY = int(360./0.2 + 1), int(180./0.2 + 1)
lon_grid = np.arange(extent_globe[0], extent_globe[1]+spacingX, spacingX)
lat_grid = np.arange(extent_globe[2], extent_globe[3]+spacingY, spacingY)
lonq, latq = np.meshgrid(lon_grid,lat_grid)

# reconstruction time steps and spacing
min_time = 0
max_time = 170
timestep_size = 1

# time array
reconstruction_times = np.arange(min_time, max_time+timestep_size, timestep_size)
# reversed (start at max_time, end at min_time)
backward_reconstruction_times = np.arange(max_time, min_time-timestep_size, -timestep_size)




# OUTPUT SAVING TOGGLES
save_output_netcdf = True # !! important
save_output_snapshots = False

# This output is useful for notebook 7
save_cumulative_subducted_carbon = True

# This is to save outputs from the parallelised "carbon_subducted_and_accreted" routine
save_outputs_to_csv = True



In [ ]:
model_dir = "./utils/Alfonso_etal_2024_modClennettMuller/"

feature_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/"
        r"*.gpml",
    )
)

rotation_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/",
        r"*.rot",
    )
)
coastlines_filename = os.path.join(
    model_dir,
    "Coastlines",
    "Clennett__etal_2020_Coastlines.gpml",
)

static_polygons = os.path.join(
    model_dir,
    "StaticPolygons/Clennett_2020_StaticPolygons.gpml"
)

model = gplately.PlateReconstruction(
    rotation_model=rotation_filenames,
    topology_features=pygplates.FeatureCollection(
        [
            i for i in pygplates.FeaturesFunctionArgument(
                feature_filenames
            ).get_features()
            if i.get_feature_type().to_qualified_string()
            != "gpml:TopologicalSlabBoundary"
        ]
        
    ),
    static_polygons=static_polygons
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=model,
    continents=coastlines_filename,
)

We need to rerun `gplately`'s `tessellate_subduction_zones` function to manually extract the orthogonal convergence rate - this is usually suppressed by default. 

In [ ]:
import os
# The large input grids are not in this repository; see the README, Data
# availability. Point CCD_SOURCE_DATA at the folder the Zenodo archive was
# unpacked into, e.g. export CCD_SOURCE_DATA=~/ccd_grids/source_data
input_dir = os.environ.get("CCD_SOURCE_DATA", "source_data")
input_cdf_filename = os.path.join(
    input_dir, "CarbonateThickness", "uncompacted_carbonate_thickness_{}Ma.nc")

In [ ]:
def latlonticks(ax):
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False,
              linewidth=1, color='gray', alpha=0.3,)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes)
    
    gl.top_labels=False
    gl.bottom_labels=False
    return

In [ ]:
def calc_vol_rate(time):
    
    # True to append convergence velocity's orthogonal and parallel components (in cm/yr)
    subduction_data = model.tessellate_subduction_zones(time, output_convergence_velocity_components=True)

    subduction_lon         = subduction_data[:,0]
    subduction_lat         = subduction_data[:,1]
    subduction_vel         = subduction_data[:,2]*1e-2
    subduction_angle       = subduction_data[:,3]
    subduction_arc_segment_len = subduction_data[:,6]
    subduction_norm        = subduction_data[:,7]
    subduction_pid_sub     = subduction_data[:,8]
    subduction_pid_over    = subduction_data[:,9]
    subduction_length      = np.deg2rad(subduction_data[:,6]) * gplately.EARTH_RADIUS * 1e3 # in metres
    subduction_convergence = np.fabs(subduction_data[:,2])*1e-2 * np.cos(np.deg2rad(subduction_data[:,3]))
    subduction_migration   = np.fabs(subduction_data[:,4])*1e-2 * np.cos(np.deg2rad(subduction_data[:,5]))
    subduction_convergence_rate_orthogonal   = subduction_data[:,10]
    
    subduction_convergence = np.clip(subduction_convergence, 0, 1e99)


    # interpolate carbonate thickness at trenches
    carbonate_grid = grids.read_netcdf_grid(
        input_cdf_filename.format(
            time
        )
    )
    # Fill all NaN continental regions with the values of their nearest neighbours
    carbonate_grid_filled = grids.fill_raster(carbonate_grid)
    
    # Sample carbon grid at subduction zones for the current timestep
    carbonate_interp, (ci, cj) = grids.sample_grid(subduction_lon,
                                          subduction_lat,
                                          carbonate_grid_filled,
                                          return_indices=True,
                                            method='nearest')
    # Thickness in m
    thickness = carbonate_interp
    
    # Trench segment length in m
    segment_length = (
        np.deg2rad(subduction_arc_segment_len)
        * gplately.EARTH_RADIUS 
        * 1000.0
    )
    # Rate of subduction in m/Myr
    subduction_rate = (
        np.array(subduction_convergence_rate_orthogonal)
        * 0.01
        * 1.0e6
    )
    # Volume of material subducted along trench segment in m^3/Myr
    volume_rate = thickness * segment_length * subduction_rate
    volume_rate = np.clip(volume_rate, 0.0, np.inf)

    print(
        (np.max(volume_rate) - np.mean(volume_rate)) / np.std(volume_rate),
        "argmax: ", np.argmax(volume_rate), "\n", 
        "max: ", 
        #np.deg2rad(subduction_arc_segment_len)[np.argmax(volume_rate)], "\n",
        thickness[np.argmax(volume_rate)], "\n", 
        segment_length[np.argmax(volume_rate)], "\n", 
        subduction_rate[np.argmax(volume_rate)] ,#, np.argmax(volume_rate)),
    thickness[np.argmax(volume_rate)] *segment_length[np.argmax(volume_rate)]*subduction_rate[np.argmax(volume_rate)],
    np.max(volume_rate))

    # Area subducted by trenches over 1 Myr (m^2). We multiply by 1e6 to ultimately
    # scale values of carbon from Megatonnes to tonnes
    subduction_surface_area = subduction_convergence * 1e6 * subduction_length

    total_vol_rate = np.sum(volume_rate*subduction_surface_area)

    return 
    
a = calc_vol_rate(0)
a

In [ ]:
import copy
def plot_subducted_carbonate_volume(time, save_fig=False):


    # True to append convergence velocity's orthogonal and parallel components (in cm/yr)
    subduction_data = model.tessellate_subduction_zones(time, output_convergence_velocity_components=True)

    subduction_lon         = subduction_data[:,0]
    subduction_lat         = subduction_data[:,1]
    subduction_vel         = subduction_data[:,2]*1e-2
    subduction_angle       = subduction_data[:,3]
    subduction_arc_segment_len = subduction_data[:,6]
    subduction_norm        = subduction_data[:,7]
    subduction_pid_sub     = subduction_data[:,8]
    subduction_pid_over    = subduction_data[:,9]
    subduction_length      = np.deg2rad(subduction_data[:,6]) * gplately.EARTH_RADIUS * 1e3 # in metres
    subduction_convergence = np.fabs(subduction_data[:,2])*1e-2 * np.cos(np.deg2rad(subduction_data[:,3]))
    subduction_migration   = np.fabs(subduction_data[:,4])*1e-2 * np.cos(np.deg2rad(subduction_data[:,5]))
    subduction_convergence_rate_orthogonal   = subduction_data[:,10]
    
    subduction_convergence = np.clip(subduction_convergence, 0, 1e99)


    # interpolate carbonate thickness at trenches
    carbonate_grid = grids.read_netcdf_grid(
        input_cdf_filename.format(
            time
        )
    )
    # Fill all NaN continental regions with the values of their nearest neighbours
    carbonate_grid_filled = grids.fill_raster(carbonate_grid)
    
    # Sample carbon grid at subduction zones for the current timestep
    carbonate_interp, (ci, cj) = grids.sample_grid(subduction_lon,
                                          subduction_lat,
                                          carbonate_grid_filled,
                                          return_indices=True,
                                            method='nearest')
    # Thickness in m
    thickness = carbonate_interp
    
    # Trench segment length in m
    segment_length = (
        np.deg2rad(subduction_arc_segment_len)
        * gplately.EARTH_RADIUS 
        * 1000.0
    )
    # Rate of subduction in m/Myr
    subduction_rate = (
        np.array(subduction_convergence_rate_orthogonal)
        * 0.01
        * 1.0e6
    )
    # Volume of material subducted along trench segment in m^3/Myr
    volume_rate = thickness * segment_length * subduction_rate
    volume_rate = np.clip(volume_rate, 0.0, np.inf)


    # Area subducted by trenches over 1 Myr (m^2). We multiply by 1e6 to ultimately
    # scale values of carbon from Megatonnes to tonnes
    subduction_surface_area = subduction_convergence * 1e6 * subduction_length

    total_vol_rate = np.sum(volume_rate*subduction_surface_area)

    proj = ccrs.Mollweide(central_longitude=60)
    fig, ax = plt.subplots(1,1, subplot_kw={'projection': proj}, figsize=(12, 7), dpi=150)

    ax.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
              transform=ccrs.PlateCarree(), zorder=0)
    
    gplot.time = time
    gplot.plot_plate_motion_vectors(ax, color='0.4', alpha=0.5, zorder=10, regrid_shape=20)
    gplot.plot_continents(ax, facecolor='0.8', edgecolor='none')


    vmin = 0
    vmax = 5e10
        
    sc = ax.scatter(subduction_lon, subduction_lat, c=volume_rate, cmap=cm.batlow, vmin=vmin, vmax=vmax,
                 transform=ccrs.PlateCarree(), rasterized=True, s=8)
    
    gplot.plot_all_topological_sections(ax, color='grey', tessellate_degrees=1)
    gplot.plot_trenches(ax, zorder=9, color='w', linewidth=0.7)
    #gplot.plot_subduction_teeth(ax, color='0.9', spacing=0.03, zorder=9)     
    latlonticks(ax)


    gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
    cax1 = fig.add_axes([0.3, 0.21, 0.25, 0.02])


    extend = "max"

    fig.colorbar(
        sc,  cax=cax1, orientation='horizontal', 
        label='Subducted carbonate volume \n(m$^3$/my)', 
        extend=extend )

    fig.subplots_adjust(bottom=0.25, top=0.95, left=0.05, right=0.80,
                    wspace=0.1, hspace=0.1)

    ax.set_title("{}Ma".format(time))


    if save_fig:
        os.makedirs(output_directory+"/subd_carbonate_vol", exist_ok=True)
        for out_format in ["png"]:
            fig.savefig(output_directory+"/subd_carbonate_vol/subd_carbonate_vol_{}Ma.{}".format(time, out_format), dpi=300, bbox_inches='tight'
                       )
    else:
        plt.show()
    plt.close()
    
    return total_vol_rate

In [ ]:
plot_subducted_carbonate_volume(0, save_fig=False)

In [ ]:
# Don't change this: directory to input files
output_directory = "./Outputs/Videos/"
os.makedirs(output_directory, exist_ok=True)

# Use LokyBackend to protect the netCDF routine
carb_sed_times = np.arange(170,-1,-1)

carbonate_thickness = Parallel(n_jobs=-1, backend='loky', verbose=1) \
(delayed(plot_subducted_carbonate_volume) \
 (reconstruction_time, 
  save_fig=True, 
 ) for reconstruction_time in carb_sed_times)

In [ ]:
import moviepy.editor as mpy

frame_list = []
for time in carb_sed_times:
    frame_list.append(
        output_directory+"/subd_carbonate_vol/subd_carbonate_vol_{}Ma.png".format(time)
    )

clip = mpy.ImageSequenceClip(frame_list, fps=25)
clip.write_videofile(output_directory+"/subd_carbonate_vol.mp4", fps=24)


In [ ]:
fig = plt.figure(figsize=(10,3.5))
ax = fig.add_subplot(111, xlabel='Age (Ma)', ylabel='Volume per Myr (m$^3$/Myr)', xlim=[170,0])

ax.plot(carb_sed_times, carbonate_thickness, color='k')
ax.set_title('Alfonso2024 - Total subducted carbonate (m$^3$/Myr)')